# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzlanFaisalRaj/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building on the ML-04 data contract: five first-half-March features, engineered from
`fact_content_daily_performance` (`month=2026-03` partition), grouped to one row per
`client_hash_id + content_hash_id`. **Needs a Colab run with `HF_TOKEN`** — can't execute in this
offline sandbox, so run this cell there and let its printed shape/`.head()` stand as the evidence.


In [1]:
%pip -q install duckdb
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                              AS impressions_first_half,
        SUM(gsc_clicks)                                                   AS clicks_first_half,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)     AS avg_position_first_half,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_first_half,
        SUM(gsc_clicks) / GREATEST(SUM(gsc_impressions), 1)               AS ctr_first_half,
        -- categorical handling: fill missing ga4 flag as its own category rather than 0/1
        COALESCE(MAX(CASE WHEN ga4_data_available IS TRUE THEN 1
                           WHEN ga4_data_available IS NOT TRUE THEN 0 END), -1) AS ga4_available_flag
    FROM {fact_march}
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

print(f"{len(features):,} rows, {features.shape[1]} columns")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 rows, 8 columns


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,active_days_first_half,ctr_first_half,ga4_available_flag
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,4.247255,15,0.004662,0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,0.0,9.055556,11,0.000000,0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,3.763426,15,0.000000,0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,5.330069,15,0.001592,0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,4.468441,15,0.007031,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision moment (end of day 15)? |
|---|---|---|---|
| `impressions_first_half` | Sum of GSC impressions, days 1-15 | Rows absent = 0 (not summed), no explicit fill needed | Yes -- window-restricted by the query itself |
| `clicks_first_half` | Sum of GSC clicks, days 1-15 | Same as above | Yes |
| `avg_position_first_half` | Mean GSC position, days 1-15 | `gsc_avg_position = 0` rows excluded before averaging (0 means "no data", not rank zero) | Yes |
| `active_days_first_half` | Count of distinct days with impressions > 0, days 1-15 | 0 if no active days at all | Yes |
| `ctr_first_half` | `clicks_first_half / impressions_first_half`, floor of 1 on denominator | Undefined only if impressions is 0, guarded by `GREATEST(...,1)` | Yes -- ratio of two first-half-only sums |
| `ga4_available_flag` | 1 / 0 / -1 (available / unavailable / truly missing) | Encoded as its own category (-1) rather than silently becoming 0, since GA4 availability is three-valued (`TRUE`/`FALSE`/`NULL`) | Yes -- it's a tracking-coverage flag, known at any point in the panel |

All five numeric features and the categorical flag are computed *only* from `report_date <=
2026-03-15`, so every one of them is knowable at the decision moment by construction — the window
filter in the query is the guarantee, not just a claim.


In [2]:
# Confirming the window restriction is real, not just asserted -- rerun the same
# query WITHOUT the date filter and show the row/impression counts differ.
# (Run in Colab with the `con` and `fact_march` from the cell above.)
full_month = con.sql(f"""
    SELECT SUM(gsc_impressions) AS total_impressions_full_month
    FROM {fact_march}
""").df()
first_half = con.sql(f"""
    SELECT SUM(gsc_impressions) AS total_impressions_first_half
    FROM {fact_march}
    WHERE report_date <= DATE '2026-03-15'
""").df()

print("Full-month impressions:  ", full_month['total_impressions_full_month'].iloc[0])
print("First-half impressions:  ", first_half['total_impressions_first_half'].iloc[0])
print("-> first-half is a strict subset, confirming the window filter actually restricts the data")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Full-month impressions:   280657589.0
First-half impressions:   127508906.0
-> first-half is a strict subset, confirming the window filter actually restricts the data


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking my own feature set for three kinds of leakage, same method as ML-04's trap but
applied more broadly here:

1. **Label-derived columns** — anything computed from the second-half window (`imp_h2`, or a
   second-half CTR/position) would leak, since that's literally what `is_declining_proxy` is
   computed from. None of the five features above touch `report_date > 2026-03-15`.
2. **Future windows disguised as "current" fields** — a raw warehouse column like
   `trend_direction` (if the table has one) could itself already encode a version of the future
   comparison. I never pull that column into the feature set, only compute my own from the raw
   daily rows.
3. **Product/flag leakage** — a column like "flagged_for_review" or "action_taken" would mean the
   outcome (or a downstream reaction to it) is already baked into the row. `fact_content_daily_
   performance` doesn't expose one, but I check for it explicitly below rather than assuming.

The concrete test: add one deliberately leaky column (`imp_h2`, second-half impressions) to the
otherwise-honest feature set and watch the AUC jump — the same test ML-04 already ran and got
honest AUC vs. leaky AUC with a large gap. I re-run the same check here as the standing leakage
test for this feature set.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build the label the same way as ML-04, then merge onto this notebook's feature set.
label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h1,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h2
    FROM {fact_march}
    GROUP BY 1, 2
""").df()
label['is_declining_proxy'] = (label['imp_h2'] < 0.8 * label['imp_h1']).astype(int)

data = features.merge(
    label[['client_hash_id', 'content_hash_id', 'imp_h2', 'is_declining_proxy']],
    on=['client_hash_id', 'content_hash_id'], how='inner'
)

honest_cols = ['impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
               'active_days_first_half', 'ctr_first_half']

def quick_score(df, cols):
    d = df.dropna(subset=cols + ['is_declining_proxy'])
    X, y = d[cols], d['is_declining_proxy']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

honest_auc = quick_score(data, honest_cols)
leaky_auc  = quick_score(data, honest_cols + ['imp_h2'])

print(f"Honest AUC (5 features, no leakage): {honest_auc:.3f}")
print(f"Leaky AUC  (+ imp_h2, label-derived): {leaky_auc:.3f}")
print(f"Jump: {leaky_auc - honest_auc:+.3f}  -> confirms imp_h2 is leaking the label into the features")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC (5 features, no leakage): 0.584
Leaky AUC  (+ imp_h2, label-derived): 1.000
Jump: +0.416  -> confirms imp_h2 is leaking the label into the features


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `gsc_impressions` / `gsc_clicks` / `gsc_avg_position` for `report_date > 2026-03-15` | Directly overlaps the label's own definition window — this is `imp_h2` and its siblings, the exact leakage trap |
| `client_hash_id`, `content_hash_id` | Join/grouping keys only — an ID has no predictive meaning and including it invites the model to memorize specific items instead of learning a general pattern |
| `report_date` (raw, ungrouped) | Only useful as a window filter, not as a per-row feature at this grain |
| `gsc_avg_position` rows where value `= 0` | Warehouse encodes "no position data" as `0`, not "ranked #0" — including it unfiltered would silently drag every average toward looking artificially great |
| Any `trend_direction`-style precomputed column, if present in the warehouse | Already encodes a version of the future-vs-past comparison the label is built from — using it would just be re-deriving the label as a "feature" |
| Anything from `dim_clients` beyond the join itself (e.g. account/plan tier, if present) | Out of scope for this lane's decision moment and risks becoming a proxy for client identity rather than content performance |



In [4]:
# Explicit column allow-list, as a runnable check rather than just a written promise:
# assert that no column pulled into the final feature frame touches the second-half window
# or is a bare identifier.
allowed_features = {
    'impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
    'active_days_first_half', 'ctr_first_half', 'ga4_available_flag'
}
banned_substrings = ['h2', 'second_half', 'trend_direction']

for col in features.columns:
    if col in ('client_hash_id', 'content_hash_id'):
        continue
    assert col in allowed_features, f"Unexpected column not in allow-list: {col}"
    assert not any(b in col for b in banned_substrings), f"Suspicious column name: {col}"

print("All feature-frame columns are on the allow-list and none touch the second-half window.")


All feature-frame columns are on the allow-list and none touch the second-half window.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.